# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate Snowflake's query behavior and SQL transformation logic for Dynamic Tables.

In [ ]:
import duckdb

# Initialize in-memory DuckDB
con = duckdb.connect(database=':memory:')

print("DuckDB Initialized.")


In [ ]:
# Simulate Silver Iceberg Tables (person, condition_occurrence)
con.execute("""
    CREATE TABLE person (
        person_source_value VARCHAR,
        birth_datetime DATE
    );
    INSERT INTO person VALUES 
        ('P1', '1980-01-01'),
        ('P2', '1990-05-15');
""")

con.execute("""
    CREATE TABLE condition_occurrence (
        person_source_value VARCHAR,
        condition_concept_id INT,
        condition_start_date DATE
    );
    INSERT INTO condition_occurrence VALUES 
        ('P1', 32020, '2023-01-10'),
        ('P1', 31981, '2023-02-15'),
        ('P2', 32020, '2023-01-20');
""")

print("Simulated Silver Tables Created.")


In [ ]:
# Simulate Silver Dynamic Table (dt_patient_conditions)
con.execute("""
    CREATE VIEW dt_patient_conditions AS
    SELECT
      p.person_source_value,
      p.birth_datetime,
      c.condition_concept_id,
      c.condition_start_date
    FROM person p
    JOIN condition_occurrence c
      ON p.person_source_value = c.person_source_value;
""")

print("Silver Dynamic Table (View) Results:")
print(con.execute("SELECT * FROM dt_patient_conditions").fetchdf())


In [ ]:
# Simulate Gold Dynamic Table (dt_monthly_conditions)
con.execute("""
    CREATE VIEW dt_monthly_conditions AS
    SELECT
      DATE_TRUNC('month', CAST(condition_start_date AS DATE)) as month,
      condition_concept_id,
      COUNT(DISTINCT person_source_value) as unique_patients
    FROM dt_patient_conditions
    GROUP BY 1, 2;
""")

print("\nGold Dynamic Table (View) Results:")
print(con.execute("SELECT * FROM dt_monthly_conditions ORDER BY month, condition_concept_id").fetchdf())
